In [0]:
sensor_bronze_df_path = "iot_krish.bronze.sensor_readings"
devices_bronze_df_path = "iot_krish.bronze.devices"
locations_bronze_df_path = "iot_krish.bronze.locations"

sensor_bronze = spark.table(sensor_bronze_df_path)
devices_bronze = spark.table(devices_bronze_df_path)
locations_bronze = spark.table(locations_bronze_df_path)

In [0]:
sensor_clean_df = (
    sensor_bronze
    .withColumn("reading_ts", to_timestamp("reading_ts"))
    .withColumn("reading_date", to_date("reading_ts"))
)

In [0]:
sensor_clean_df = sensor_clean_df.withColumn(
    "is_anomaly",
    when((col("temperature_c") > 90) | (col("vibration_hz") > 3.0), True).otherwise(False)
)

In [0]:
from pyspark.sql.window import Window

rolling_window = Window.partitionBy("device_id").orderBy("reading_ts").rowsBetween(-2, 0)

sensor_clean_df = sensor_clean_df.withColumn(
    "rolling_avg_temp",
    avg("temperature_c").over(rolling_window)
)

In [0]:
sensor_clean_join_df = sensor_clean_df \
    .join(devices_bronze, "device_id", "left") \
    .join(locations_bronze, "location_id", "left") \
    .select(
        sensor_clean_df["*"],
        devices_bronze["device_type"],
        devices_bronze["maintenance_due"],
        locations_bronze["zone_name"],
        locations_bronze["floor"],
        locations_bronze["supervisor"]
    )

In [0]:
(sensor_clean_join_df.write
 .format("delta")
 .mode("overwrite")
 .partitionBy("location_id")
 .option("overwriteSchema", "true")
 .saveAsTable("iot_krish.silver.sensor_readings"))

In [0]:
%sql
OPTIMIZE iot_krish.silver.sensor_readings
ZORDER BY (device_id, reading_ts);